# Consigna del desafio 3

- Realizar un modelo de lenguaje con tokenización por caracteres
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.

### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import requests
import torchinfo
from sklearn import metrics

## Datos

In [2]:
url = "https://www.gutenberg.org/ebooks/2000.txt.utf-8"

response = requests.get(url)
text = ''

if response.status_code == 200:
    text = response.text

text = text.lower()

In [3]:
text[:100]

'\ufeffthe project gutenberg ebook of don quijote\r\n    \r\nthis ebook is for the use of anyone anywhere in t'

In [4]:
first_chapter = text.find("capítulo primero")
text = text[first_chapter:]


In [5]:
text[:1000]

'capítulo primero. que trata de la condición y ejercicio del famoso hidalgo\r\ndon quijote de la mancha\r\n\r\nen un lugar de la mancha, de cuyo nombre no quiero acordarme, no ha mucho\r\ntiempo que vivía un hidalgo de los de lanza en astillero, adarga antigua,\r\nrocín flaco y galgo corredor. una olla de algo más vaca que carnero,\r\nsalpicón las más noches, duelos y quebrantos los sábados, lantejas los\r\nviernes, algún palomino de añadidura los domingos, consumían las tres\r\npartes de su hacienda. el resto della concluían sayo de velarte, calzas de\r\nvelludo para las fiestas, con sus pantuflos de lo mesmo, y los días de\r\nentresemana se honraba con su vellorí de lo más fino. tenía en su casa una\r\nama que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte,\r\ny un mozo de campo y plaza, que así ensillaba el rocín como tomaba la\r\npodadera. frisaba la edad de nuestro hidalgo con los cincuenta años; era de\r\ncomplexión recia, seco de carnes, enjuto de rostro, gran

## Elegir tamaño del contexto

In [6]:
max_context_size = 100

## Tokenizado

In [7]:
class CharacterTokenizer:

    def __init__(self, text):
        self.chars = sorted(list(set(text)))

        self.vocab_size = len(self.chars)

        self.char2idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx2char = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, text_to_code):
        return [self.char2idx[c] for c in text_to_code]
    
    def decode(self, idx_lst):
        return [self.idx2char[i] for i in idx_lst]

In [8]:
tokenizer = CharacterTokenizer(text)
tokenized_text = tokenizer.encode(text)

In [9]:
tokenized_text[:1000]

[31,
 29,
 44,
 62,
 48,
 49,
 40,
 43,
 2,
 44,
 46,
 37,
 41,
 33,
 46,
 43,
 13,
 2,
 45,
 49,
 33,
 2,
 48,
 46,
 29,
 48,
 29,
 2,
 32,
 33,
 2,
 40,
 29,
 2,
 31,
 43,
 42,
 32,
 37,
 31,
 37,
 65,
 42,
 2,
 53,
 2,
 33,
 38,
 33,
 46,
 31,
 37,
 31,
 37,
 43,
 2,
 32,
 33,
 40,
 2,
 34,
 29,
 41,
 43,
 47,
 43,
 2,
 36,
 37,
 32,
 29,
 40,
 35,
 43,
 1,
 0,
 32,
 43,
 42,
 2,
 45,
 49,
 37,
 38,
 43,
 48,
 33,
 2,
 32,
 33,
 2,
 40,
 29,
 2,
 41,
 29,
 42,
 31,
 36,
 29,
 1,
 0,
 1,
 0,
 33,
 42,
 2,
 49,
 42,
 2,
 40,
 49,
 35,
 29,
 46,
 2,
 32,
 33,
 2,
 40,
 29,
 2,
 41,
 29,
 42,
 31,
 36,
 29,
 11,
 2,
 32,
 33,
 2,
 31,
 49,
 53,
 43,
 2,
 42,
 43,
 41,
 30,
 46,
 33,
 2,
 42,
 43,
 2,
 45,
 49,
 37,
 33,
 46,
 43,
 2,
 29,
 31,
 43,
 46,
 32,
 29,
 46,
 41,
 33,
 11,
 2,
 42,
 43,
 2,
 36,
 29,
 2,
 41,
 49,
 31,
 36,
 43,
 1,
 0,
 48,
 37,
 33,
 41,
 44,
 43,
 2,
 45,
 49,
 33,
 2,
 50,
 37,
 50,
 62,
 29,
 2,
 49,
 42,
 2,
 36,
 37,
 32,
 29,
 40,
 35,
 43,
 2,
 32,
 3

## Estructuracion del dataset

In [10]:
p_val = 0.9

X = []
y = []

for i in range(len(tokenized_text) - max_context_size):
    X.append(tokenized_text[i : i + max_context_size - 1])
    y.append(tokenized_text[i + 1 : i + max_context_size])


In [11]:
X_train = X[:int(len(X) * p_val)]
y_train = y[:int(len(X) * p_val)]

In [12]:
X_val = X[int(len(X) * p_val):]
y_val = y[int(len(X) * p_val):]


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
class CharDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.x = torch.tensor(X)
        self.y = torch.tensor(y, dtype = torch.long)

    def __len__(self):
        return self.x.shape[0]
    
    def __getitem__(self, index):
        return self.x[index], self.y[index]

In [15]:
batch_size = 64

In [16]:
train_dataset = CharDataset(X_train, y_train)
val_dataset = CharDataset(X_val, y_val)
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)


## Modelo RNN

In [ ]:
class CharModelRnn(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.rnn_size = 64
        self.num_layers = 2
        self.hidden_size = 200
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = self.rnn_size)
        self.rnn = nn.RNN(input_size = self.rnn_size, hidden_size = self.hidden_size, num_layers = self.num_layers, batch_first = True, dropout = 0.3)
        self.fc = nn.Linear(in_features = 200, out_features = vocab_size)

    def forward(self, x, prev_state = None):
        if prev_state is None:
            batch_size = x.shape[0] #(batch, seq_size)
            prev_state = self.init_hidden(batch_size)
    
        x = self.embedding(x)
        rnn_output, _ = self.rnn(x)
        x = self.fc(rnn_output)


        return x
    
    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device) #h_0
               



## Modelo LSTM

In [ ]:
class CharModelLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.lstm_size = 64
        self.num_layers = 2
        self.hidden_size = 200
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = self.lstm_size)
        self.rnn = nn.LSTM(input_size=self.lstm_size, hidden_size=self.hidden_size, batch_first=True,
                            num_layers=self.num_layers, dropout=0.3)
        self.fc = nn.Linear(in_features = 200, out_features = vocab_size)

    def forward(self, x, prev_state = None):
        if prev_state is None:
            batch_size = x.shape[0] #(batch, seq_size)
            prev_state = self.init_hidden(batch_size)
    
        x = self.embedding(x)
        rnn_output, (_, _)= self.rnn(x)
        x = self.fc(rnn_output)


        return x
    
    def init_hidden(self, batch_size):
        return (torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device),
                torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)) #h_0 y c_0
               



## Modelo GRU

In [ ]:
class CharModelGRU(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.gru_size = 64
        self.num_layers = 2
        self.hidden_size = 200
        self.embedding = nn.Embedding(num_embeddings = vocab_size, embedding_dim = self.gru_size)
        self.rnn = nn.GRU(input_size=self.gru_size, hidden_size=self.hidden_size, batch_first=True,
                            num_layers=self.num_layers, dropout=0.3)
        self.fc = nn.Linear(in_features = 200, out_features = vocab_size)

    def forward(self, x, prev_state = None):
        if prev_state is None:
            batch_size = x.shape[0] #(batch, seq_size)
            prev_state = self.init_hidden(batch_size)
    
        x = self.embedding(x)
        rnn_output, _ = self.rnn(x)
        x = self.fc(rnn_output)

        return x
    
    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)#h_0

In [72]:
rnn_model = CharModelRnn(vocab_size = tokenizer.vocab_size)
lstm_model = CharModelLSTM(vocab_size = tokenizer.vocab_size)
gru_model = CharModelGRU(vocab_size = tokenizer.vocab_size)


In [73]:
torchinfo.summary(rnn_model, input_size = (batch_size, max_context_size), dtypes=[torch.long])


RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Embedding: 1, RNN: 1]

In [ ]:
torchinfo.summary(lstm_model, input_size = (batch_size, max_context_size), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
CharModelLSTM                            [64, 100, 76]             --
├─Embedding: 1-1                         [64, 100, 64]             4,864
├─LSTM: 1-2                              [64, 100, 200]            534,400
├─Linear: 1-3                            [64, 100, 76]             15,276
Total params: 554,540
Trainable params: 554,540
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 3.42
Input size (MB): 0.05
Forward/backward pass size (MB): 17.41
Params size (MB): 2.22
Estimated Total Size (MB): 19.68

In [59]:
torchinfo.summary(gru_model, input_size = (batch_size, max_context_size), dtypes=[torch.long])


RuntimeError: Failed to run torchinfo. See above stack traces for more details. Executed layers up to: [Embedding: 1, GRU: 1]

## Entrenamiento

In [24]:
from torchmetrics.classification import Accuracy

def train(model, train_loader, valid_loader, epochs, loss_function, optimizer, model_name):
    train_loss = []
    train_accuracy = []
    valid_loss = []
    valid_accuracy = []
    perplexity = []
    patience  = 3
    min_score = np.inf
    patience_conter = 0

    accuracy_metric = Accuracy(task="multiclass", num_classes=tokenizer.vocab_size).to(device)

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        accuracy_metric.reset()

        for train_data, train_target in train_loader:
            train_data = train_data.to(device)
            train_target = train_target.to(device)
            optimizer.zero_grad()

            epoch_y_hat = model(train_data)
            predictions_reshaped = epoch_y_hat.view(-1, tokenizer.vocab_size)
            targets_reshaped = train_target.view(-1)


            loss = loss_function(predictions_reshaped, targets_reshaped)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() 
            accuracy_metric.update(predictions_reshaped, targets_reshaped)

        epoch_train_loss = total_train_loss / len(train_loader)
        epoch_train_accuracy = accuracy_metric.compute()
        train_loss.append(epoch_train_loss)
        train_accuracy.append(epoch_train_accuracy.cpu().numpy())

        model.eval()
        total_valid_loss = 0
        accuracy_metric.reset() 

        with torch.no_grad(): # Desactiva el cálculo de gradientes
            for valid_data, valid_target in valid_loader:
                valid_data = valid_data.to(device)
                valid_target = valid_target.to(device)
                
                epoch_y_hat = model(valid_data)

                predictions_reshaped = epoch_y_hat.view(-1, tokenizer.vocab_size)
                targets_reshaped = valid_target.view(-1)

                loss = loss_function(predictions_reshaped, targets_reshaped)
                total_valid_loss += loss.item()
                accuracy_metric.update(predictions_reshaped, targets_reshaped)

        # 5. Calcular métricas finales de la época
        epoch_valid_loss = total_valid_loss / len(valid_loader)
        epoch_valid_accuracy = accuracy_metric.compute()
        valid_loss.append(epoch_valid_loss)
        valid_accuracy.append(epoch_valid_accuracy.cpu().numpy())
        
        epoch_perplexity = torch.exp(torch.tensor(epoch_valid_loss))
        perplexity.append(epoch_perplexity.item())

        print(f" Epoca: {epoch} | " \
          f"Train/Valid loss: {train_loss[-1]}/{valid_loss[-1]}) | " \
          f"Train/Valid acc: {train_accuracy[-1]}/{valid_accuracy[-1]}) | " \
          f"Perpelexity: {perplexity[-1]}")
    

        if perplexity[-1] < min_score:
            min_score = perplexity[-1]
            patience_conter = 0
            torch.save(model.state_dict(), model_name)
            print("Saved new model!")
        else: 
            patience_conter += 1
        if patience_conter == patience:
            break

        
    return {
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'valid_loss': valid_loss,
        'valid_accuracy': valid_accuracy,
        'perplexity': perplexity
    }

In [25]:
loss_function = nn.CrossEntropyLoss()
optimezer = torch.optim.RMSprop(params = rnn_model.parameters(), lr = 0.001)
rnn_model = rnn_model.to(device)
rnn_model_name = 'rnn_model.pth'
train_rnn_result = train(
    model = rnn_model,
    train_loader = train_dataloader,
    valid_loader = val_dataloader,
    epochs = 10,
    loss_function = loss_function,
    optimizer = optimezer,
    model_name = rnn_model_name
)


 Epoca: 0 | Train/Valid loss: 1.3764813060694823/1.6604687335007744) | Train/Valid acc: 0.5634753704071045/0.5374338626861572) | Perpelexity: 5.261776447296143
Saved new model!
 Epoca: 1 | Train/Valid loss: 1.3252619215603063/1.665484115642761) | Train/Valid acc: 0.5776737928390503/0.5400162935256958) | Perpelexity: 5.288232326507568
 Epoca: 2 | Train/Valid loss: 1.3174908718294298/1.6630501933978639) | Train/Valid acc: 0.579691469669342/0.5422416925430298) | Perpelexity: 5.27537727355957
 Epoca: 3 | Train/Valid loss: 1.3137443561083382/1.6658322900964035) | Train/Valid acc: 0.5806868672370911/0.5409360527992249) | Perpelexity: 5.290074348449707


In [26]:
optimezer = torch.optim.RMSprop(params = lstm_model.parameters(), lr = 0.0005)
lstm_model_name = 'lstm_model.pth'

train_lstm_result = train(
    model = lstm_model,
    train_loader = train_dataloader,
    valid_loader = val_dataloader,
    epochs = 10,
    loss_function = loss_function,
    optimizer = optimezer,
    model_name = lstm_model_name
)


 Epoca: 0 | Train/Valid loss: 1.2514632082119377/1.5894105661431805) | Train/Valid acc: 0.5979692339897156/0.5703772902488708) | Perpelexity: 4.900859355926514
Saved new model!
 Epoca: 1 | Train/Valid loss: 1.1207035341501292/1.6187435674036166) | Train/Valid acc: 0.6345512270927429/0.5748328566551208) | Perpelexity: 5.046745300292969
 Epoca: 2 | Train/Valid loss: 1.0934812904213231/1.6406026066568928) | Train/Valid acc: 0.6426319479942322/0.5742517113685608) | Perpelexity: 5.1582770347595215
 Epoca: 3 | Train/Valid loss: 1.079941864509909/1.6606359846181675) | Train/Valid acc: 0.6467064023017883/0.5739556550979614) | Perpelexity: 5.2626566886901855


In [28]:
optimezer = torch.optim.RMSprop(params = gru_model.parameters(), lr = 0.0005)
gru_model_name = 'gru_model.pth'
train_gru_result = train(
    model = gru_model,
    train_loader = train_dataloader,
    valid_loader = val_dataloader,
    epochs = 10,
    loss_function = loss_function,
    optimizer = optimezer,
    model_name = gru_model_name
)

 Epoca: 0 | Train/Valid loss: 1.272799935662248/1.6242034171319037) | Train/Valid acc: 0.5932924747467041/0.5636711120605469) | Perpelexity: 5.074375629425049
Saved new model!
 Epoca: 1 | Train/Valid loss: 1.1809359196693923/1.635230780329228) | Train/Valid acc: 0.6193430423736572/0.5651279091835022) | Perpelexity: 5.130641937255859
 Epoca: 2 | Train/Valid loss: 1.1633372302519713/1.6358019404749864) | Train/Valid acc: 0.6244065165519714/0.563646137714386) | Perpelexity: 5.133573055267334
 Epoca: 3 | Train/Valid loss: 1.1542276900737498/1.6357711236184254) | Train/Valid acc: 0.6270114183425903/0.5673700571060181) | Perpelexity: 5.133415222167969


## Gready Search

In [29]:
def pad_sequence_pre(sequence, max_len, pad_value=0):
    """
    Pre-pad a sequence to max_len with pad_value.
    """
    # Ensure sequence is a list of integers
    if not isinstance(sequence, (list, tuple)):
        raise ValueError(f"Sequence must be a list or tuple, got {type(sequence)}")
    if not all(isinstance(x, int) for x in sequence):
        raise ValueError("Sequence must contain integers")
    
    # Convert to tensor
    tensor = torch.tensor(sequence, dtype=torch.long)
    pad_size = max_len - len(tensor)
    
    if pad_size > 0:
        padding = torch.full((pad_size,), pad_value, dtype=torch.long)
        tensor = torch.cat([padding, tensor])
    elif pad_size < 0:
        tensor = tensor[-max_len:]  # Truncate if too long
    return tensor

In [30]:
def gready_search(model, input_text, tokenizer, max_length, n_words):
    
    model.eval()
    output_text = input_text

    for _ in range(n_words):
        encoded = tokenizer.encode(output_text.lower())
        encoded = pad_sequence_pre(encoded, max_length)
        y_hat = model(encoded.unsqueeze(0).to(device))
        
        predict = np.argmax(y_hat[0, -1, :].cpu().detach().numpy()).item()

        out_word = ''
        out_word = tokenizer.decode([predict])[0]

        output_text += out_word

    return output_text


In [35]:
import torch
import torch.nn.functional as F
import numpy as np

def select_candidates(preds, num_beams, vocab_size, history_probs, history_tokens, temp, mode):
    pred_large = []
    for idx, pp in enumerate(preds):
        pred_large.append(torch.log(pp + 1e-10) + history_probs[idx])
    pred_large = torch.cat(pred_large, dim=0).cpu().numpy()

    if mode == 'det':
        idx_select = np.argsort(pred_large)[::-1][:num_beams]
    elif mode == 'sto':
        probs = F.softmax(torch.tensor(pred_large) / temp, dim=0).numpy()
        idx_select = np.random.choice(np.arange(len(pred_large)), num_beams, p=probs)
    else:
        raise ValueError("Mode must be 'det' or 'sto'.")

    new_history_tokens = np.concatenate((np.array(history_tokens)[idx_select // vocab_size],
                                         np.array([idx_select % vocab_size]).T), axis=1)
    return pred_large[idx_select], new_history_tokens

def beam_search(model, num_beams, num_words, input_text, tokenizer, max_context_size, temp=1.0, mode='det', device='cuda'):
    model.eval()
    encoded = tokenizer.encode(input_text.lower())
    #print("Encoded input:", encoded, len(encoded))  # Debug: Check encoded output
    if not encoded:
        raise ValueError("Encoded input is empty. Check input_text or tokenizer.")
    
    # Pad the initial sequence
    padded = pad_sequence_pre(encoded, max_context_size - 1)
    #print("Padded input:", padded, padded.shape)  # Debug: Should be [99]
    
    # Add batch dimension
    input_tensor = padded.unsqueeze(0).to(device)  # Shape: [1, 99]
    #print("Input tensor shape:", input_tensor.shape)  # Debug: Should be [1, 99]
    
    history_tokens = [padded.cpu().numpy()] * num_beams  # Convert to numpy for consistency
    history_probs = [0.0] * num_beams

    # First prediction
    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]  # Shape: [1, vocab_size]
        probs = F.softmax(logits, dim=-1)[0]    # Shape: [vocab_size]

    history_probs, history_tokens = select_candidates([probs], num_beams, tokenizer.vocab_size,
                                                     history_probs, history_tokens, temp, mode)

    # Beam search loop
    for _ in range(num_words - 1):
        preds = []
        for hist in history_tokens:
            input_tensor = torch.tensor([hist[-(max_context_size - 1):]], dtype=torch.long).to(device)
            #print("History tensor shape:", input_tensor.shape)  # Debug: Should be [1, 99]
            with torch.no_grad():
                logits = model(input_tensor)[:, -1, :]
                probs = F.softmax(logits, dim=-1)[0]
            preds.append(probs)
        history_probs, history_tokens = select_candidates(preds, num_beams, tokenizer.vocab_size,
                                                         history_probs, history_tokens, temp, mode)

    # Decode best beam
    best_seq = history_tokens[0]
    decoded = tokenizer.decode(best_seq.tolist())
    #print("Decoded tokens:", decoded)  # Debug
    return decoded

In [42]:
gready_search(rnn_model, "en un lugar de", tokenizer, 50, 20)

'en un lugar de la mancha, y a la m'

In [37]:
output = beam_search(
    model=rnn_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=1.0,
    mode='det',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', '.', '\r', '\n', '\r', '\n', '—', ' ', 'n', 'o', ' ']


In [38]:
output = beam_search(
    model=rnn_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=0.5,
    mode='sto',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', ',', ' ', 'p', 'o', 'r', 'q', 'u', 'e', ' ', 'a']


In [39]:
output = beam_search(
    model=rnn_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=10,
    mode='sto',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', 't', 'a', 'c', 'i', 'h', '»', 's', ';', 'y', 't']


In [49]:
gready_search(lstm_model, "en un lugar de", tokenizer, 50, 20)


'en un lugar de qué temer tantas ve'

In [50]:
gready_search(gru_model, "en un lugar de", tokenizer, 50, 20)


'en un lugar de la caballería andan'

In [51]:
output = beam_search(
    model=lstm_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=1.0,
    mode='det',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', '.', '\r', '\n', '\r', '\n', '—', ' ', 'p', 'u', 'e']


In [52]:
output = beam_search(
    model=lstm_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=0.5,
    mode='sto',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', ',', ' ', 'q', 'u', 'e', ' ', 'y', 'o', ' ', 'p']


In [53]:
output = beam_search(
    model=lstm_model,
    num_beams=5,
    num_words=10,
    input_text="en un lugar de la mancha",
    tokenizer=tokenizer,
    max_context_size=max_context_size,
    temp=10,
    mode='sto',
    device=device
)
print("Generated text:", output)

Generated text: ['\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', '\n', 'e', 'n', ' ', 'u', 'n', ' ', 'l', 'u', 'g', 'a', 'r', ' ', 'd', 'e', ' ', 'l', 'a', ' ', 'm', 'a', 'n', 'c', 'h', 'a', ' ', 'v', 'á', ' ', '—', '.', ' ', '¿', 'c', 'r']
